In [1]:
from attack.attack_manager import *
from utils.eval_util import SettingsRandomizer
from tqdm import tqdm
#physical_devices = tf.config.list_physical_devices('GPU') 
#for device in physical_devices:
#    tf.config.experimental.set_memory_growth(device, True)

In [2]:
curr_dir = os.getcwd()
settings = load_yaml(os.path.join(curr_dir, 'configs', 'config_tjunction.yaml'))


In [4]:
settings['ego_vehicle']['spawn_transform'] = [106.7, 129.7, 0.5, 0., 180., 0.]
settings_randomizer = SettingsRandomizer(settings)
settings_randomizer.attacker_spawn_range = [83., 85.2, 145., 120.]
settings_randomizer.victim_spawn_range = [83., 85.2, 125., 145.]
settings_randomizer.ego_speed_range = [0., 0.]
settings_randomizer.ego_speed_number = 1
settings_randomizer.victim_spawn_number = 120
#settings_randomizer.attacker_spawn_number = 12
settings_list = settings_randomizer.generate_baseline_vehicle()

In [5]:
attack_manager = AttackManager(settings)

In [ ]:
# os.system('taskkill /f /im CarlaUE4-Win64-Shipping.exe')
# os.startfile(r'C:\Research\Carla\WindowsNoEditor\CarlaUE4.exe')
# time.sleep(10.0)
base_path = r'/home/jiaruili/Documents/exp/advTraj/baselines/perpendicular_baseline'
attack_manager.attacker_speed_limit = 3
attack_manager.attacker_movement_limit = 0.3
attack_manager.lr = 1
attack_manager.iter = 20
#attack_manager.threshold = 0.3
#bboxes = attack_manager.run_attack(settings)
num_success = 0
num = 0
for setting in tqdm(settings_list[66:]):
    setting['output_settings']['img_dir'] = os.path.join(base_path, 'imgs')
    setting['output_settings']['bbox_dir'] = os.path.join(base_path, 'bboxes')
    setting['output_settings']['config_dir'] = os.path.join(base_path, 'configs')
    setting['output_settings']['traj_dir'] = os.path.join(base_path, 'trajs')
    try:
        success = attack_manager.collect_baseline(setting, higher_view=False, save_img_bbox=True)
    except:
        #os.system('taskkill /f /im CarlaUE4-Win64-Shipping.exe')
        #os.startfile(r'C:\Research\Carla\WindowsNoEditor\CarlaUE4.exe')
        #time.sleep(15.0)
        continue
    if success:
        num_success += 1
    num += 1
    print("Result: {} / {}".format(num_success, num))
    if num >= 35:
        break
print("ASR: {}".format(num_success/len(settings_list)))